PyTorch 损失函数

损失函数的核心作用确实是衡量模型预测结果与真实目标之间的差异，但它不只是计算一个误差数值，更重要的是为模型的参数优化提供方向和依据。

可以把 PyTorch 训练过程中的三个核心角色理解为：
- 损失函数 criterion： 告诉模型当前预测得有多差。
- 反向传播 loss.backward()： 计算损失对模型参数的梯度，告诉我们参数应该往哪个方向调整才能降低损失。
- 优化器 optimizer： 根据梯度和学习率，实际更新模型参数。


In [2]:
import torch
from torch import nn

In [3]:
# 训练流程
# 损失函数既不是优化器，也不负责更新参数：
# 输入 x -> model(x) 得到预测 -> loss(prediction, target) -> backward 得到梯度 -> optimizer.step 更新参数
# 训练的目标通常是通过优化器调整参数，使训练数据上的损失降低。对于一个 batch，损失一般先对样本求和或平均，得到标量，再调用 `loss.backward()` 计算梯度。

In [24]:
## 3. 从一个简单误差开始
# 假设真实值为 3，模型预测为 2：
target = torch.tensor([3.0], requires_grad=True)
prediction = torch.tensor([2.0])
criterion = nn.MSELoss()
loss = criterion(prediction, target)
print(loss)

print(target.grad)
loss.backward() # 反向传播计算梯度
print(target.grad)


target.grad.zero_()
target = torch.tensor([3.0, 2.0], requires_grad=True)
prediction = torch.tensor([2.0, 1.0])
loss = criterion(prediction, target)
loss.backward()
print(target.grad)


tensor(1., grad_fn=<MseLossBackward0>)
None
tensor([2.])
tensor([1., 1.])


In [25]:
### 4.2 使用函数形式的loss
import torch.nn.functional as F

target = torch.tensor([3.0, 2.0], requires_grad=True)
prediction = torch.tensor([2.0, 1.0])
loss = F.mse_loss(prediction, target)
loss.backward()
print(target.grad)


tensor([1., 1.])


In [27]:
## 5. 最重要的通用参数：`reduction`
# 常用的 reduction 操作：mean、min、max

target = torch.tensor([3.0, 2.0], requires_grad=True)
prediction = torch.tensor([2.0, 1.0])
criterion = nn.L1Loss(reduction='sum')
loss = criterion(prediction, target)
loss.backward()
print(target.mean())
print(target.grad)

tensor(2.5000, grad_fn=<MeanBackward0>)
tensor([1., 1.])



损失函数的选择
| 任务 | 模型输出 | target 常见格式 | 推荐起点 |
| --- | --- | --- | --- |
| 单目标/多目标回归 | 任意实数 | 浮点数，形状与输出相同 | `MSELoss` 或 `L1Loss` |
| 回归且关注异常值鲁棒性 | 任意实数 | 浮点数 | `SmoothL1Loss` / `HuberLoss` |
| 二分类 | 每样本一个 logit | 0/1 浮点数，形状与 logit 相同 | `BCEWithLogitsLoss` |
| 多标签分类 | 每类别一个 logit | 多个独立 0/1 浮点标签 | `BCEWithLogitsLoss` |
| 单标签多分类 | 每类别一个 logit | 类别索引 `long` | `CrossEntropyLoss` |
| 多分类且标签为软分布 | 每类别一个 logit | 与 logits 同形状的概率分布 | `CrossEntropyLoss` 概率目标或 KL 散度，依场景选择 |
| 逐像素分割 | 每像素每类别一个 logit | 类别索引图 `long` | `CrossEntropyLoss` |
| 二值分割 | 每像素一个 logit | 0/1 浮点 mask | `BCEWithLogitsLoss` |
| 语言模型 next-token | 每 token 每词表一个 logit | token 类别索引 `long` | `CrossEntropyLoss(ignore_index=...)` |

快速选择
- 回归：MSELoss
- 二分类：BCEWithLogitsLoss
- 多分类、语言模型：CrossEntropyLoss


In [33]:
# 快速选择
# - 回归：MSELoss
# - 二分类、多标签：BCEWithLogitsLoss
# - 多分类、语言模型：CrossEntropyLoss

## 7. 回归损失
### 7.1 均方误差 `MSELoss`
### 7.2 平均绝对误差 `L1Loss`
### 7.3 `SmoothL1Loss` 与 `HuberLoss`

## 8. 二分类与多标签：`BCEWithLogitsLoss`
# Binary Cross Entropy 描述二元目标的概率似然。训练时推荐使用 `nn.BCEWithLogitsLoss`，它把 sigmoid 和 BCE 合并，以提高数值稳定性。
### 8.1 二分类与多标签的区别
# - 二分类：一个样本属于正类或负类，输出 1 个 logit。
# - 多标签：一个样本可以同时拥有多个标签，每个类别独立判断，输出 `C` 个 logits。
# - 单标签多分类：一个样本只属于 `C` 个互斥类别之一，通常使用 `CrossEntropyLoss`，不是对每类独立做 BCE。

# 多标签的列子
model = nn.Linear(20, 3)  # 三个标签可以同时为 1， [20, 3] 表示一个数据有20个特征数据，并且是有三个标签
criterion = nn.BCEWithLogitsLoss()
logits = model(torch.randn(5, 20)) # [5, 20] 表示 一个批次输入5个样本，每个样本有20个特征数据
# 表示 5 个数据分别带有什么标签
targets = torch.tensor([[1., 0., 1.], [0., 1., 0.], [1., 1., 0.], [0., 0., 0.], [1., 0., 0.]])
# 损失计算
loss = criterion(logits, targets)
print(loss)
print(logits)
# 通过sigmod计算概率，先将原始概率转化为sigmod概率分布
probabilities = torch.sigmoid(logits)
predicted_class = (probabilities >= 0.5).long() # 默认概率超过0.5就是这个标签
print(predicted_class)

tensor(0.7630, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[ 0.3635,  0.5810, -0.3122],
        [ 0.1884,  0.4944,  0.3150],
        [-1.0263, -0.2233, -0.9222],
        [ 0.1794, -0.0693,  0.1077],
        [-0.3903,  0.0083, -0.1459]], grad_fn=<AddmmBackward0>)
tensor([[1, 1, 0],
        [1, 1, 1],
        [0, 0, 0],
        [1, 0, 1],
        [0, 1, 0]])


In [34]:
target = torch.randint(0, 3, (32,), dtype=torch.long)
target

tensor([2, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2, 0, 0, 2, 2, 2, 1, 1, 1, 0, 1, 1, 0, 0,
        0, 1, 1, 2, 0, 2, 1, 2])